In [0]:
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("domain_name", "")
dbutils.widgets.text("sub_domain_name", "")

table_name = dbutils.widgets.get("table_name").strip()
domain_name = dbutils.widgets.get("domain_name").strip()
sub_domain_name = dbutils.widgets.get("sub_domain_name").strip()

# Used only for case-insensitive metadata lookup
table_name_lower = table_name.lower()
domain_name_lower = domain_name.lower()
sub_domain_name_lower = sub_domain_name.lower()

print("INPUT")
print(f"table_name     : {table_name}")
print(f"domain_name    : {domain_name}")
print(f"sub_domain    : {sub_domain_name}")

INPUT
table_name     : 01_TXN_transaction_lifecycle
domain_name    : TXN_PROCESSING
sub_domain    : TRANSACTION_LIFECYCLE_STATUS


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
import json
import time


CATALOG = "mastercard_card_241614"
SCHEMA = "default"

SCHEMA_VALIDATION_NOTEBOOK = "schema_validation"
RULE_EXECUTION_NOTEBOOK = "rule_execution"


EXECUTIONS_TABLE = f"{CATALOG}.{SCHEMA}.executions_m"
ASSET_EXECUTIONS_TABLE = f"{CATALOG}.{SCHEMA}.asset_executions_m"
RESULTS_TABLE = f"{CATALOG}.{SCHEMA}.dq_execution_results_m"


print("Configuration loaded")

print(f"Schema validation notebook : {SCHEMA_VALIDATION_NOTEBOOK}")
print(f"Rule execution notebook    : {RULE_EXECUTION_NOTEBOOK}")

print(f"Executions table            : {EXECUTIONS_TABLE}")
print(f"Asset executions table      : {ASSET_EXECUTIONS_TABLE}")
print(f"Results table               : {RESULTS_TABLE}")

Configuration loaded
Schema validation notebook : schema_validation
Rule execution notebook    : rule_execution
Executions table            : mastercard_card_241614.default.executions_m
Asset executions table      : mastercard_card_241614.default.asset_executions_m
Results table               : mastercard_card_241614.default.dq_execution_results_m


In [0]:
if not table_name:
    raise Exception("table_name is required.")

if not domain_name:
    raise Exception("domain_name is required.")

if not sub_domain_name:
    raise Exception("sub_domain_name is required.")

print("Input validation PASSED")

Input validation PASSED


In [0]:
assets_df = spark.table(
    f"{CATALOG}.{SCHEMA}.assets_m"
)

domains_df = spark.table(
    f"{CATALOG}.{SCHEMA}.domains_m"
)

sub_domains_df = spark.table(
    f"{CATALOG}.{SCHEMA}.sub_domains_m"
)

In [0]:
asset_df = (
    assets_df
    .filter(
        F.col("is_active") == True
    )
    .filter(
        F.lower(
            F.regexp_extract(
                F.col("s3_path"),
                r"([^/]+)$",
                1
            )
        ) == table_name_lower
    )
)

In [0]:
asset_count = asset_df.count()

if asset_count == 0:
    raise Exception(
        f"No active asset found for table_name='{table_name}'."
    )

if asset_count > 1:
    raise Exception(
        f"Multiple active assets found for table_name='{table_name}'."
    )

asset = asset_df.first()

asset_id = asset["asset_id"]
asset_domain_id = asset["domain_id"]
asset_sub_domain_id = asset["sub_domain_id"]

print("ASSET RESOLVED")
print(f"asset_id         : {asset_id}")
print(f"asset_name       : {asset['asset_name']}")
print(f"domain_id        : {asset_domain_id}")
print(f"sub_domain_id    : {asset_sub_domain_id}")
print(f"s3_path          : {asset['s3_path']}")

ASSET RESOLVED
asset_id         : 28
asset_name       : 01_TXN_transaction_lifecycle
domain_id        : 1
sub_domain_id    : 1
s3_path          : source/TXN_PROCESSING/TRANSACTION_LIFECYCLE_STATUS/01_txn_transaction_lifecycle


In [0]:
domain_df = (
    domains_df
    .filter(
        F.lower(F.col("domain_name")) == domain_name_lower
    )
    .filter(
        F.col("is_active") == True
    )
)

if domain_df.count() != 1:
    raise Exception(
        f"Domain '{domain_name}' could not be uniquely resolved."
    )

domain = domain_df.first()

domain_id = domain["domain_id"]

sub_domain_df = (
    sub_domains_df
    .filter(
        F.col("domain_id") == domain_id
    )
    .filter(
        F.lower(F.col("sub_domain_name")) == sub_domain_name_lower
    )
    .filter(
        F.col("is_active") == True
    )
)

if sub_domain_df.count() != 1:
    raise Exception(
        f"Sub-domain '{sub_domain_name}' could not be uniquely resolved "
        f"under domain '{domain_name}'."
    )

sub_domain = sub_domain_df.first()

sub_domain_id = sub_domain["sub_domain_id"]

In [0]:
if asset_domain_id != domain_id:
    raise Exception(
        f"Domain mismatch. "
        f"Asset domain_id={asset_domain_id}, "
        f"requested domain_id={domain_id}."
    )

if asset_sub_domain_id != sub_domain_id:
    raise Exception(
        f"Sub-domain mismatch. "
        f"Asset sub_domain_id={asset_sub_domain_id}, "
        f"requested sub_domain_id={sub_domain_id}."
    )

print("ASSET / DOMAIN VALIDATION PASSED")
print(f"domain_id       : {domain_id}")
print(f"sub_domain_id   : {sub_domain_id}")

ASSET / DOMAIN VALIDATION PASSED
domain_id       : 1
sub_domain_id   : 1


In [0]:
schema_validation_result = dbutils.notebook.run(
    SCHEMA_VALIDATION_NOTEBOOK,
    0,
    {
        "table_name": table_name,
        "domain_name": domain_name,
        "sub_domain_name": sub_domain_name
    }
)

print(
    f"Schema validation result: {schema_validation_result}"
)

Schema validation result: PASS


In [0]:
if schema_validation_result != "PASS":
    raise Exception(
        f"Schema validation did not return PASS. "
        f"Result='{schema_validation_result}'."
    )

print("SCHEMA VALIDATION PASSED")
print("Proceeding to DQ rule execution.")

SCHEMA VALIDATION PASSED
Proceeding to DQ rule execution.


In [0]:
print("====================================")
print("CREATING EXECUTION")
print("====================================")


spark.sql(f"""
    INSERT INTO {EXECUTIONS_TABLE}
    (
        execution_type,
        pipeline_id,
        triggered_by,
        status,
        started_at,
        completed_at
    )
    VALUES
    (
        'DQ_RUN',
        NULL,
        NULL,
        'RUNNING',
        current_timestamp(),
        NULL
    )
""")


print("Execution record inserted.")

CREATING EXECUTION
Execution record inserted.


In [0]:
latest_execution = (
    spark.table(EXECUTIONS_TABLE)
    .filter(F.col("execution_type") == "DQ_RUN")
    .filter(F.col("status") == "RUNNING")
    .orderBy(F.col("created_at").desc())
    .select(
        "execution_id",
        "execution_type",
        "status",
        "started_at",
        "created_at"
    )
    .first()
)


if latest_execution is None:
    raise RuntimeError(
        "Execution was inserted, but execution_id "
        "could not be retrieved."
    )


execution_id = latest_execution["execution_id"]


print("")
print("EXECUTION CREATED")
print("------------------------------------")
print(f"execution_id : {execution_id}")
print(f"status       : {latest_execution['status']}")
print(f"started_at   : {latest_execution['started_at']}")
print(f"created_at   : {latest_execution['created_at']}")


EXECUTION CREATED
------------------------------------
execution_id : 2
status       : RUNNING
started_at   : 2026-10-01 13:35:40.343957
created_at   : 2026-10-01 13:35:38.995692


In [0]:
print("====================================")
print("ASSET EXECUTION TABLE SCHEMA")
print("====================================")

spark.table(ASSET_EXECUTIONS_TABLE).printSchema()

ASSET EXECUTION TABLE SCHEMA
root
 |-- asset_execution_id: long (nullable = false)
 |-- execution_id: long (nullable = false)
 |-- asset_id: long (nullable = false)
 |-- status: string (nullable = false)
 |-- started_at: timestamp (nullable = true)
 |-- completed_at: timestamp (nullable = true)
 |-- total_rules: integer (nullable = true)
 |-- passed_rules: integer (nullable = true)
 |-- failed_rules: integer (nullable = true)
 |-- total_records: long (nullable = true)
 |-- passed_records: long (nullable = true)
 |-- failed_records: long (nullable = true)
 |-- dq_score_cumulative: decimal(5,2) (nullable = true)
 |-- error_message: string (nullable = true)
 |-- created_at: timestamp (nullable = false)
 |-- dimension: string (nullable = false)
 |-- threshold_value_cumulative: decimal(18,6) (nullable = true)



In [0]:
print("====================================")
print("CREATING ASSET EXECUTION")
print("====================================")


spark.sql(f"""
    INSERT INTO {ASSET_EXECUTIONS_TABLE}
    (
        execution_id,
        asset_id,
        status,
        started_at,
        completed_at,
        total_rules,
        passed_rules,
        failed_rules,
        total_records,
        passed_records,
        failed_records,
        dq_score_cumulative,
        error_message,
        dimension,
        threshold_value_cumulative
    )
    VALUES
    (
        {execution_id},
        {asset_id},
        'RUNNING',
        current_timestamp(),
        NULL,
        0,
        0,
        0,
        0,
        0,
        0,
        NULL,
        NULL,
        'ALL',
        NULL
    )
""")


print("Asset execution record inserted.")


# --------------------------------------------------
# Retrieve generated asset_execution_id
# --------------------------------------------------

latest_asset_execution = (
    spark.table(ASSET_EXECUTIONS_TABLE)
    .filter(F.col("execution_id") == execution_id)
    .filter(F.col("asset_id") == asset_id)
    .filter(F.col("status") == "RUNNING")
    .orderBy(F.col("created_at").desc())
    .select(
        "asset_execution_id",
        "execution_id",
        "asset_id",
        "status",
        "started_at",
        "created_at"
    )
    .first()
)


if latest_asset_execution is None:
    raise RuntimeError(
        "Asset execution was inserted, but "
        "asset_execution_id could not be retrieved."
    )


asset_execution_id = (
    latest_asset_execution["asset_execution_id"]
)


print("")
print("ASSET EXECUTION CREATED")
print("------------------------------------")
print(
    f"asset_execution_id : "
    f"{asset_execution_id}"
)
print(
    f"execution_id       : "
    f"{latest_asset_execution['execution_id']}"
)
print(
    f"asset_id           : "
    f"{latest_asset_execution['asset_id']}"
)
print(
    f"status             : "
    f"{latest_asset_execution['status']}"
)

CREATING ASSET EXECUTION
Asset execution record inserted.

ASSET EXECUTION CREATED
------------------------------------
asset_execution_id : 2
execution_id       : 2
asset_id           : 28
status             : RUNNING


In [0]:
rules_df = spark.table(
    f"{CATALOG}.{SCHEMA}.rules_m"
)

rules_assignments_df = spark.table(
    f"{CATALOG}.{SCHEMA}.rules_assignments_m"
)

asset_columns_df = spark.table(
    f"{CATALOG}.{SCHEMA}.asset_columns_m"
)

In [0]:
rules_to_execute_df = (
    rules_assignments_df.alias("ra")

    .join(
        rules_df.alias("r"),
        F.col("ra.rule_code") == F.col("r.rule_code"),
        "inner"
    )

    .join(
        asset_columns_df.alias("ac"),
        (
            (F.col("ra.target_column_id") == F.col("ac.column_id"))
            &
            (F.col("ra.asset_id") == F.col("ac.asset_id"))
        ),
        "left"
    )

    .filter(
        F.col("ra.asset_id") == asset_id
    )

    .filter(
        F.col("ra.is_enabled") == True
    )

    .filter(
        F.col("ra.is_current") == True
    )

    .filter(
        F.col("ra.effective_from") <= F.current_timestamp()
    )

    .filter(
        F.col("ra.effective_to").isNull()
        |
        (F.col("ra.effective_to") > F.current_timestamp())
    )

    .filter(
        F.col("r.is_active") == True
    )

    .select(
        F.col("ra.assignment_id"),
        F.col("ra.asset_id"),

        F.col("ra.rule_code"),
        F.col("ra.version_number"),

        F.col("ra.target_column_id"),
        F.col("ac.column_name").alias("target_column_name"),

        F.col("r.dimension"),
        F.col("r.rule_name"),
        F.col("r.rule_type"),

        F.coalesce(
            F.col("ra.target_type"),
            F.col("r.target_type")
        ).alias("target_type"),

        F.col("r.parameter_definition"),

        F.col("ra.parameter_values"),
        F.col("ra.sql_expression"),
        F.col("ra.threshold_value"),
        F.col("ra.evaluation_operation"),

        F.col("ra.target_asset_id"),
        F.col("ra.target_asset_column_id")
    )
)

In [0]:
rule_count = rules_to_execute_df.count()

print(f"ACTIVE RULES ASSIGMENTS FOUND: {rule_count}")

ACTIVE RULES ASSIGMENTS FOUND: 23


In [0]:
display(
    rules_to_execute_df.orderBy(
        "dimension",
        "rule_code",
        "version_number"
    )
)

assignment_id,asset_id,rule_code,version_number,target_column_id,target_column_name,dimension,rule_name,rule_type,target_type,parameter_definition,parameter_values,sql_expression,threshold_value,evaluation_operation,target_asset_id,target_asset_column_id
84,28,C01,1,254,null,COMPLETENESS,Completeness %,SQL,COLUMN / TABLE,null,null,null,95.000000,>=,null,null
62,28,C01,1,269,DQ_FLAG,COMPLETENESS,Completeness %,SQL,COLUMN / TABLE,null,null,null,95.000000,>=,null,null
10,28,C05,1,null,null,COMPLETENESS,Record completeness %,PARAMETERIZED,TABLE,"{ ""parameters"":[ { ""name"":""required_columns"", ""label"":""Required columns"", ""control"":""column_multi_select"", ""required"":true } ] }",null,null,95.000000,>=,null,null
12,28,C07,1,null,null,COMPLETENESS,Selected-column completeness %,PARAMETERIZED,TABLE,"{ ""parameters"":[ { ""name"":""selected_columns"", ""label"":""Selected columns"", ""control"":""column_multi_select"", ""required"":true } ] }",null,null,95.000000,>=,null,null
66,28,K03,1,253,null,CONSISTENCY,Standardization compliance %,PARAMETERIZED,COLUMN,"{ ""parameters"":[ { ""name"":""canonical_representation"", ""label"":""Canonical representation"", ""control"":""text"", ""required"":true } ] }",null,null,98.000000,>=,null,null
88,28,K03,1,251,null,CONSISTENCY,Standardization compliance %,PARAMETERIZED,COLUMN,"{ ""parameters"":[ { ""name"":""canonical_representation"", ""label"":""Canonical representation"", ""control"":""text"", ""required"":true } ] }",null,null,98.000000,>=,null,null
57,28,X03,1,null,null,CUSTOM,Custom Tabular SQL Rule,CUSTOM_SQL,TABLE,null,null,null,null,null,null,null
58,28,X04,1,null,null,CUSTOM,Custom SQL Single-Result Rule,CUSTOM_SQL,TABLE / COLUMN,null,null,null,null,null,null,null
13,28,I04,1,null,null,INTEGRITY,Primary-key null %,PARAMETERIZED,TABLE,"{ ""parameters"":[ { ""name"":""primary_key_column"", ""label"":""Primary key column"", ""control"":""column_select"", ""required"":true } ] }",null,null,0.000000,<=,null,null
19,28,P01,1,null,null,PROFILING,Row count,SQL,TABLE,null,null,null,null,null,null,null


In [0]:
if rule_count == 0:

    print(
        f"No active DQ rules are assigned to asset_id={asset_id}."
    )

    dbutils.notebook.exit("NO_RULES")

In [0]:
print("====================================")
print("STARTING DQ RULE EXECUTION")
print("====================================")

rule_results = []

total_rules = rules_to_execute_df.count()

passed_rules = 0
failed_rules = 0
error_rules = 0

total_records = 0
passed_records = 0
failed_records = 0


# for rule in rules_to_execute_df.collect():
for rule in (
    rules_to_execute_df
    .filter(F.col("assignment_id") == 62)
    .collect()
):

    assignment_id = rule["assignment_id"]
    current_rule_code = rule["rule_code"]

    print("")
    print("====================================")
    print(f"EXECUTING RULE: {current_rule_code}")
    print(f"ASSIGNMENT ID : {assignment_id}")
    print("====================================")


    try:

        result = dbutils.notebook.run(
            RULE_EXECUTION_NOTEBOOK,
            0,
            {
                "asset_id": str(rule["asset_id"]),

                "assignment_id": str(
                    rule["assignment_id"]
                ),

                "rule_code": str(
                    rule["rule_code"]
                ),

                "version_number": str(
                    rule["version_number"]
                ),

                "target_column_id": (
                    str(rule["target_column_id"])
                    if rule["target_column_id"] is not None
                    else ""
                ),

                "parameter_values": (
                    rule["parameter_values"]
                    if rule["parameter_values"] is not None
                    else ""
                ),

                "sql_expression": (
                    rule["sql_expression"]
                    if rule["sql_expression"] is not None
                    else ""
                ),

                "threshold_value": (
                    str(rule["threshold_value"])
                    if rule["threshold_value"] is not None
                    else ""
                ),

                "evaluation_operation": (
                    rule["evaluation_operation"]
                    if rule["evaluation_operation"] is not None
                    else ""
                ),

                "asset_execution_id": str(
                    asset_execution_id
                )
            }
        )


        print("")
        print("BOOK 3 RESULT")
        print("------------------------------------")
        print(result)


        result_json = json.loads(result)


        rule_status = result_json.get(
            "status"
        )


        rule_results.append(
            result_json
        )


        # ------------------------------------------
        # Aggregate rule status
        # ------------------------------------------

        if rule_status == "PASS":

            passed_rules += 1

        elif rule_status == "FAIL":

            failed_rules += 1

        else:

            error_rules += 1


        # ------------------------------------------
        # Aggregate record metrics
        # ------------------------------------------

        total_records += int(
            result_json.get(
                "records_evaluated",
                0
            ) or 0
        )

        passed_records += int(
            result_json.get(
                "records_passed",
                0
            ) or 0
        )

        failed_records += int(
            result_json.get(
                "records_failed",
                0
            ) or 0
        )


    except Exception as e:

        error_rules += 1

        error_result = {
            "status": "ERROR",
            "rule_code": current_rule_code,
            "assignment_id": assignment_id,
            "error": str(e)
        }

        rule_results.append(
            error_result
        )

        print("")
        print(
            f"RULE EXECUTION ERROR: "
            f"{current_rule_code}"
        )
        print(str(e))


print("")
print("====================================")
print("DQ RULE EXECUTION COMPLETED")
print("====================================")

print(f"Total rules   : {total_rules}")
print(f"Passed rules  : {passed_rules}")
print(f"Failed rules  : {failed_rules}")
print(f"Error rules   : {error_rules}")

print("")
print(f"Total records  : {total_records}")
print(f"Passed records : {passed_records}")
print(f"Failed records : {failed_records}")

STARTING DQ RULE EXECUTION

EXECUTING RULE: C01
ASSIGNMENT ID : 62

BOOK 3 RESULT
------------------------------------
{"status": "PASS", "rule_code": "C01", "assignment_id": 62, "version_number": 1, "metric_value": 100.0, "threshold_value": 95.0, "evaluation_operation": ">=", "records_evaluated": 100, "records_passed": 100, "records_failed": 0, "failure_percentage": 0.0, "result_written": true}

DQ RULE EXECUTION COMPLETED
Total rules   : 23
Passed rules  : 1
Failed rules  : 0
Error rules   : 0

Total records  : 100
Passed records : 100
Failed records : 0


In [0]:
# assets_df = assets_df.filter(
#     F.lower(F.col("asset_name")) == table_name_lower
# )

# assets_df = assets_df.filter(
#     F.col("is_active") == True
# )

# assets_df.display()